In [1]:
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore")

In [2]:
from src.helper import get_split_data

X_trn, y_trn, X_val, y_val, X_tst, y_tst = get_split_data.split_data_for_training(3,'data/preprocessed/preprocessed_1.csv',17)

print(X_trn.shape)
print(X_val.shape)
print(X_tst.shape)

(1268, 45)
(10, 45)
(10, 45)


In [3]:
X_trn

,points_home,points_away,home_last_team_goal,home_last_team_shoton,home_last_team_possession,away_last_team_goal,away_last_team_shoton,away_last_team_possession,team_strength_home,team_strength_away,...,rolling_avg_shoton_diff,rolling_stability_shoton_diff,points_diff,rolling_avg_goals_ratio,rolling_stability_goal_ratio,rolling_avg_goals_conversion_rate_ratio,rolling_stability_goals_conversion_rate_ratio,rolling_avg_shoton_ratio,rolling_stability_shoton_ratio,points_ratio
0,0,0,3.000000,9.0,40.0,1.50,5.0,52.0,74.781818,67.438889,...,4.60,0.431634,0,0.810219,1.173078,0.261583,0.265656,2.393939,1.221748,0.000000
1,0,0,1.666667,8.0,48.0,2.00,2.0,60.0,68.266919,66.648485,...,-0.30,-0.889766,0,1.126168,1.240039,0.850172,0.398121,0.946429,0.686290,0.000000
2,0,0,1.000000,7.0,54.0,2.00,9.0,37.0,71.960354,71.607828,...,-4.30,-0.954972,0,1.029167,0.682424,2.273314,1.785853,0.494118,0.702120,0.000000
3,0,0,2.000000,11.0,47.0,1.00,7.0,54.0,69.084848,52.832071,...,-0.40,0.560703,0,0.832031,0.769734,0.992160,1.312755,0.935484,1.267304,0.000000
4,0,0,2.000000,10.0,40.0,3.50,9.0,52.0,60.648196,58.133045,...,-2.60,-1.456867,0,0.681481,0.630295,1.000930,0.788665,0.679012,0.680305,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1263,19,29,1.000000,4.0,41.0,3.00,5.0,59.0,70.545455,70.709091,...,-5.20,-1.785235,-10,0.483384,0.219067,1.292220,1.167562,0.415730,0.535427,0.655172
1264,14,29,2.000000,7.0,57.0,1.00,9.0,54.0,63.981818,58.636364,...,-0.90,-0.157576,-15,0.837302,0.440476,0.845687,0.629439,0.850000,0.938971,0.482759
1265,18,25,1.000000,1.0,73.0,3.25,11.0,56.0,73.027273,76.100000,...,-5.00,-0.405329,-7,0.539683,0.305054,1.662020,1.975087,0.444444,0.862316,0.720000
1266,10,23,1.000000,6.0,46.0,1.00,9.0,59.0,77.318182,59.590909,...,-1.95,-1.148942,-13,1.314286,1.394305,2.124070,3.669741,0.775862,0.740259,0.434783


In [4]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif = [variance_inflation_factor(X_trn.values, i) for i in range(X_trn.shape[1])]
print(np.round(vif, 2))

[   inf    inf   9.22   9.82  28.95   8.97   8.87  28.57    inf    inf
    inf    inf    inf    inf    inf    inf    inf   5.27   4.67    inf
    inf    inf    inf    inf    inf    inf    inf    inf    inf    inf
    inf    inf    inf    inf    inf    inf    inf    inf 180.69  13.9
 116.88  15.61 127.16  21.92   4.8 ]


In [5]:
correlation_matrix = pd.DataFrame(X_trn.values).corr().abs()

high_corr_var = np.where(correlation_matrix > 0.7)

corr_pairs = [(correlation_matrix.columns[x], correlation_matrix.columns[y])
              for x, y in zip(*high_corr_var) if x != y and x < y]

print("Highly correlated pairs:")
for pair in corr_pairs:
    print(pair)

columns_to_drop = []
for pair in corr_pairs:
    if pair[1] not in columns_to_drop:
        columns_to_drop.append(pair[1])

columns_to_drop = [X_trn.columns[i] for i in columns_to_drop if i < len(X_trn.columns)]
X_trn_reduced = X_trn.drop(columns=columns_to_drop)
X_val_reduced = X_val.drop(columns=columns_to_drop)
X_tst_reduced = X_tst.drop(columns=columns_to_drop)

print("Shapes:")
print(X_trn_reduced.shape)
print(X_val_reduced.shape)
print(X_tst_reduced.shape)

Highly correlated pairs:
(0, 1)
(8, 10)
(14, 16)
(19, 20)
(20, 32)
(21, 22)
(21, 33)
(21, 40)
(22, 34)
(22, 41)
(23, 35)
(24, 36)
(25, 26)
(25, 31)
(26, 32)
(27, 28)
(27, 33)
(28, 34)
(29, 35)
(30, 36)
(31, 32)
(31, 38)
(32, 38)
(32, 39)
(33, 34)
(33, 40)
(33, 41)
(34, 40)
(34, 41)
(35, 42)
(36, 43)
(38, 39)
(40, 41)
Shapes:
(1268, 26)
(10, 26)
(10, 26)


In [6]:
X_trn_reduced.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1268 entries, 0 to 1267
Data columns (total 26 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   points_home                             1268 non-null   int64  
 1   home_last_team_goal                     1268 non-null   float64
 2   home_last_team_shoton                   1268 non-null   float64
 3   home_last_team_possession               1268 non-null   float64
 4   away_last_team_goal                     1268 non-null   float64
 5   away_last_team_shoton                   1268 non-null   float64
 6   away_last_team_possession               1268 non-null   float64
 7   team_strength_home                      1268 non-null   float64
 8   team_strength_away                      1268 non-null   float64
 9   team_aggression_home                    1268 non-null   float64
 10  team_aggression_away                    1268 non-null   floa

In [7]:
def calculate_vif(X):
    vif_data = pd.DataFrame()
    vif_data["Variable"] = X.columns
    vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    return vif_data

X_trn_vif = X_trn.copy()
X_vif_val = X_val.copy()
X_vif_tst = X_tst.copy()

threshold = 2.5
max_iterations = 100

for i in range(max_iterations):
    vif_data = calculate_vif(X_trn_vif)

    if (vif_data['VIF'] > threshold).any() or (np.isinf(vif_data['VIF'])).any():
        max_vif_var = vif_data.loc[vif_data['VIF'].idxmax(), 'Variable']
        print(f"Dropping {max_vif_var} with VIF = {vif_data['VIF'].max()}")

        X_trn_vif = X_trn_vif.drop(max_vif_var, axis=1)
        X_vif_val = X_vif_val.drop(max_vif_var, axis=1)
        X_vif_tst = X_vif_tst.drop(max_vif_var, axis=1)
    else:
        break

print("\nFinal VIF values:")
print(calculate_vif(X_trn_vif))
print(f"\nRemaining features: {X_trn_vif.shape[1]} out of {X_trn.shape[1]}")

Dropping points_home with VIF = inf
Dropping team_strength_home with VIF = inf
Dropping team_aggression_home with VIF = inf
Dropping team_acceleration_home with VIF = inf
Dropping rolling_avg_goals_home with VIF = inf
Dropping rolling_stability_goal_home with VIF = inf
Dropping rolling_avg_goals_conversion_rate_home with VIF = inf
Dropping rolling_stability_goals_conversion_rate_home with VIF = inf
Dropping rolling_avg_shoton_home with VIF = inf
Dropping rolling_stability_shoton_home with VIF = inf
Dropping rolling_avg_shoton_away with VIF = 470.50887844572276
Dropping team_strength_away with VIF = 369.95770257448487
Dropping team_acceleration_away with VIF = 205.78322715958933
Dropping rolling_avg_goals_ratio with VIF = 155.91357375085977
Dropping rolling_avg_goals_away with VIF = 119.80824588163877
Dropping rolling_avg_shoton_ratio with VIF = 100.68742706454611
Dropping team_aggression_away with VIF = 85.72175983356037
Dropping rolling_avg_goals_conversion_rate_ratio with VIF = 77.93

In [8]:
X_trn_vif.to_csv('../../data/featureselection/X_trn_vif.csv', index=False)
X_vif_val.to_csv('../../data/featureselection/X_vif_val.csv', index=False)
X_vif_tst.to_csv('../../data/featureselection/X_vif_tst.csv', index=False)
y_trn.to_csv('../../data/featureselection/y_trn.csv', index=False)
y_val.to_csv('../../data/featureselection/y_val.csv', index=False)
y_tst.to_csv('../../data/featureselection/y_tst.csv', index=False)

In [9]:
X_trn_reduced.to_csv('../../data/featureselection/X_trn_reduced.csv', index=False)
X_val_reduced.to_csv('../../data/featureselection/X_val_reduced.csv', index=False)
X_tst_reduced.to_csv('../../data/featureselection/X_tst_reduced.csv', index=False)
y_trn.to_csv('../../data/featureselection/y_trn.csv', index=False)
y_val.to_csv('../../data/featureselection/y_val.csv', index=False)
y_tst.to_csv('../../data/featureselection/y_tst.csv', index=False)